# Detectar la fase **antes** de modelar

## De dónde viene

`05_TopDown_share_modelado` mostró que la fase del producto es el mejor
discriminador que encontramos: el top-down gana **+15,7%** en lanzamiento y
**+7,2%** en crecimiento, y pierde **−4,8%** en meseta. O sea que la fase no es una
feature más — decide qué arquitectura conviene.

Este notebook la convierte en una **etapa explícita previa** a la regresión, y le
suma el contexto de mercado.

## Fase del producto × fase de su mercado

Mirar sólo al producto pierde la mitad de la historia. Un producto que cae dentro de
una categoría que crece está **perdiendo una pelea competitiva**. El mismo producto
cayendo dentro de una categoría que cae está simplemente **siguiendo a su mercado**.
Misma serie, dos situaciones distintas, y probablemente dos dinámicas futuras
distintas.

Se cruzan las dos dimensiones en un **estado conjunto**.

## La sutileza que define el diseño

Predecimos `tn(t+2)`. Lo que importa entonces no es la fase en `t` sino **la que va
a tener en `t+2`**. Eso convierte la etapa 1 en un **clasificador**, no en una simple
etiqueta.

Y abre la pregunta previa: ¿hace falta el clasificador? Si las fases son *pegajosas*
— si la de `t` casi siempre sigue siendo la de `t+2` — alcanza con la observada y el
clasificador sólo agrega error. Eso se mide con la **matriz de transición** antes de
entrenar nada.

## Las arquitecturas que se comparan

| | etapa 1 | etapa 2 |
|---|---|---|
| **A** baseline | — | un GBM sobre todo |
| **B** fase observada | fase en `t` (producto y mercado) como feature | un GBM |
| **C** fase predicha | clasificador de la fase en `t+2`, **out-of-fold** | un GBM con esa predicción |
| **D** un modelo por fase | fase en `t` como router | un GBM por fase |
| **E** ruteo de arquitectura | fase en `t` | top-down o bottom-up según la fase |

Todas con la misma partición, las mismas features base y semilla fija.

> Se arrastran las tres correcciones de reproducibilidad de `05`: sumas
> redondeadas, `sort` antes de cada `shift`, y LightGBM determinístico. Sin eso las
> diferencias entre arquitecturas quedan tapadas por el ruido.

## 0 — Ambiente

In [ ]:
import os, json, warnings
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore")


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env:
        return Path(env).expanduser().resolve()
    for cand in ("/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_OUT = BUCKET / "datasets_fe"
DIR_OUT.mkdir(parents=True, exist_ok=True)

SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ   = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
CMAP  = LinearSegmentedColormap.from_list("azul", SEQ)
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRILLA, EJE_C, FONDO = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO,
    "axes.edgecolor": EJE_C, "axes.labelcolor": TINTA2,
    "text.color": TINTA, "xtick.color": MUDO, "ytick.color": MUDO,
    "grid.color": GRILLA, "grid.linewidth": .8,
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "figure.dpi": 110,
    "legend.frameon": False,
})

def limpiar(ax, titulo=None, y=None, x=None):
    if titulo: ax.set_title(titulo, color=TINTA, loc="left", pad=10)
    if y: ax.set_ylabel(y)
    if x: ax.set_xlabel(x)
    ax.grid(axis="x", visible=False)
    return ax

H = 2
MESES_TRAIN_FIN = 201905
MESES_VAL  = [201907, 201908]
MESES_TEST = [201910]
SEMILLA = 102191

FASES  = {0:"lanzamiento", 1:"crecimiento", 2:"meseta", 3:"caida"}
MFASES = {0:"mercado_cae", 1:"mercado_estable", 2:"mercado_crece"}
print(f"BUCKET: {BUCKET}")

## 1 — Panel, fase del producto y fase del mercado

Las dos fases se calculan igual: pendiente relativa a 3 meses sobre la media móvil,
escalada por el nivel propio. Todo **expansivo** — el pico es el máximo hasta ese
mes, nunca el de toda la serie.

In [ ]:
sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t").unique(subset=["product_id"])

def a_m(c): return (pl.col(c) // 100) * 12 + (pl.col(c) % 100)
def m_a_periodo(m): return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1
def periodo_a_m(p): return (p // 100) * 12 + (p % 100)

# .round(6): group_by().sum() en float no es bit-reproducible entre hilos.
base = (sell.group_by(["product_id","periodo"]).agg(pl.col("tn").sum().round(6).alias("tn"))
            .with_columns(a_m("periodo").alias("m")))
nac = base.group_by("product_id").agg(pl.col("m").min().alias("m_nace"),
                                      pl.col("m").max().alias("m_ult"))
grilla = (nac.select("product_id","m_nace","m_ult")
             .with_columns(pl.int_ranges("m_nace", pl.col("m_ult")+1).alias("m"))
             .explode("m").drop("m_nace","m_ult"))

P = (grilla.join(base, on=["product_id","m"], how="left")
           .with_columns(pl.col("tn").fill_null(0.0))
           .join(prod.select("product_id","cat1","cat2","cat3","brand"), on="product_id", how="left")
           .join(nac.select("product_id","m_nace"), on="product_id", how="left")
           .with_columns((pl.col("m") - pl.col("m_nace")).alias("edad"))
           .sort(["product_id","m"]))

CT = (P.group_by(["cat3","m"]).agg(pl.col("tn").sum().round(6).alias("tn_cat3"),
                                   pl.col("product_id").n_unique().alias("n_comp"))
        .sort(["cat3","m"]))
P = (P.join(CT, on=["cat3","m"], how="left").sort(["product_id","m"])
       .with_columns(pl.when(pl.col("tn_cat3") > 0)
                       .then(pl.col("tn")/pl.col("tn_cat3")).otherwise(0.0).alias("share")))
print(f"panel: {P.height:,} filas · {P['product_id'].n_unique()} productos · {P['cat3'].n_unique()} cat3")

In [ ]:
U = 0.10      # umbral de pendiente relativa

# ── Fase del PRODUCTO (expansiva) ────────────────────────────────────────
P = P.sort(["product_id","m"]).with_columns([
    pl.col("tn").rolling_mean(3, min_periods=1).over("product_id").alias("tn_ma3"),
    pl.col("tn").cum_max().over("product_id").alias("pico_hist"),
])
P = P.with_columns(
    ((pl.col("tn_ma3") - pl.col("tn_ma3").shift(3).over("product_id")) /
      pl.when(pl.col("pico_hist") > 0).then(pl.col("pico_hist")).otherwise(1.0)).alias("pend3"))
P = P.with_columns(
    pl.when(pl.col("edad") <= 2).then(pl.lit(0))
     .when(pl.col("pend3") >  U).then(pl.lit(1))
     .when(pl.col("pend3") < -U).then(pl.lit(3))
     .otherwise(pl.lit(2)).alias("fase"))

# ── Fase del MERCADO (misma logica sobre el total de la cat3) ────────────
CTf = CT.sort(["cat3","m"]).with_columns([
    pl.col("tn_cat3").rolling_mean(3, min_periods=1).over("cat3").alias("cat3_ma3"),
    pl.col("tn_cat3").cum_max().over("cat3").alias("cat3_pico"),
])
CTf = CTf.with_columns(
    ((pl.col("cat3_ma3") - pl.col("cat3_ma3").shift(3).over("cat3")) /
      pl.when(pl.col("cat3_pico") > 0).then(pl.col("cat3_pico")).otherwise(1.0)).alias("cat3_pend3"))
CTf = CTf.with_columns(
    pl.when(pl.col("cat3_pend3") < -U/2).then(pl.lit(0))
     .when(pl.col("cat3_pend3") >  U/2).then(pl.lit(2))
     .otherwise(pl.lit(1)).alias("fase_mercado"))

P = P.join(CTf.select("cat3","m","cat3_ma3","cat3_pend3","fase_mercado"),
           on=["cat3","m"], how="left").sort(["product_id","m"])

print(P.group_by("fase").len().sort("fase").with_columns(
        pl.col("fase").replace_strict(FASES).alias("nombre")))
print()
print(P.group_by("fase_mercado").len().sort("fase_mercado").with_columns(
        pl.col("fase_mercado").replace_strict(MFASES).alias("nombre")))

## 2 — El estado conjunto

La celda de abajo cruza las dos dimensiones. Lo que hay que mirar no es la
frecuencia de cada celda sino **si el error de predicción difiere entre ellas**: si
el WAPE naive es parecido en todas, el estado conjunto no aporta y alcanza con la
fase del producto sola.

La celda interesante es **producto en caída dentro de mercado que crece** — el caso
competitivo puro.

In [ ]:
P = P.with_columns(pl.col("tn").shift(-H).over("product_id").alias("y_tn"))
Q = P.drop_nulls("y_tn")

M = np.full((4, 3), np.nan); N = np.zeros((4, 3), int)
for (f, fm), g in Q.group_by(["fase","fase_mercado"]):
    if g.height < 30:
        continue
    r = g["y_tn"].to_numpy(); p = np.maximum(g["tn"].to_numpy(), 0)
    den = np.abs(r).sum()
    M[f, fm] = np.abs(r-p).sum()/den if den else np.nan
    N[f, fm] = g.height

fig, ax = plt.subplots(figsize=(6.4, 4.2))
im = ax.imshow(M, cmap=CMAP, aspect="auto")
ax.set_xticks(range(3)); ax.set_xticklabels([MFASES[i] for i in range(3)], fontsize=8)
ax.set_yticks(range(4)); ax.set_yticklabels([FASES[i] for i in range(4)], fontsize=8)
for i in range(4):
    for j in range(3):
        if np.isfinite(M[i, j]):
            ax.text(j, i, f"{M[i,j]:.3f}\nn={N[i,j]}", ha="center", va="center", fontsize=8,
                    color="#ffffff" if M[i,j] > np.nanmax(M)*.6 else TINTA)
ax.grid(False)
ax.set_title("WAPE naive por estado conjunto  ·  fase del producto × fase del mercado",
             color=TINTA, loc="left", pad=10, fontsize=9)
plt.tight_layout(); plt.show()

print("Mas oscuro = mas dificil de predecir.")
print(f"rango de WAPE entre celdas: {np.nanmin(M):.3f} - {np.nanmax(M):.3f}")
print("Si el rango es amplio, el estado conjunto separa regimenes de dificultad")
print("distinta y vale la pena pasarlo al modelo.")

## 3 — ¿Hacen falta un clasificador? La matriz de transición

`P(fase en t+2 | fase en t)`. Si la diagonal domina, la fase es pegajosa y la
observada en `t` ya es un buen proxy — el clasificador sólo agregaría error.

In [ ]:
P = P.with_columns(pl.col("fase").shift(-H).over("product_id").alias("fase_fut"))
T = P.drop_nulls("fase_fut")

TM = np.zeros((4, 4))
for (a, b), g in T.group_by(["fase","fase_fut"]):
    TM[a, b] = g.height
TM = TM / np.maximum(TM.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(5.6, 4.2))
ax.imshow(TM, cmap=CMAP, aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_xticklabels([FASES[i] for i in range(4)], rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(4)); ax.set_yticklabels([FASES[i] for i in range(4)], fontsize=8)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{TM[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="#ffffff" if TM[i,j] > .55 else TINTA)
ax.grid(False)
ax.set_ylabel(f"fase en t"); ax.set_xlabel(f"fase en t+{H}")
ax.set_title("Matriz de transición de fase", color=TINTA, loc="left", pad=10)
plt.tight_layout(); plt.show()

persistencia = float(np.mean([TM[i, i] for i in range(4)]))
acierto_persistir = float((T["fase"] == T["fase_fut"]).mean())
print(f"diagonal media           : {persistencia:.3f}")
print(f"acierto de 'la fase no cambia': {acierto_persistir:.3f}")
print()
if acierto_persistir > .6:
    print("La fase es PEGAJOSA: la observada en t es buen proxy de la de t+2.")
    print("Un clasificador tiene poco margen sobre simplemente persistir.")
else:
    print("La fase ROTA bastante: vale la pena un clasificador que la anticipe,")
    print("porque usar la observada en t seria usar informacion desactualizada.")

## 4 — Features y partición

In [ ]:
P = P.sort(["product_id","m"]).with_columns([
    *[pl.col("tn").shift(k).over("product_id").alias(f"tn_lag{k}") for k in range(1, 7)],
    *[pl.col("share").shift(k).over("product_id").alias(f"share_lag{k}") for k in range(1, 4)],
    pl.col("tn").rolling_mean(12, min_periods=3).over("product_id").alias("tn_ma12"),
    pl.col("tn").rolling_std(12, min_periods=3).over("product_id").alias("tn_sd12"),
    pl.col("share").rolling_mean(3, min_periods=1).over("product_id").alias("share_ma3"),
    pl.col("share").rolling_mean(6, min_periods=2).over("product_id").alias("share_ma6"),
    *[pl.col("tn_cat3").shift(k).over("product_id").alias(f"cat3_lag{k}") for k in (1,2,3)],
    ((pl.col("m")-1) % 12 + 1).alias("mes_cal"),
])
P = P.with_columns([
    ((pl.col("tn") - pl.col("tn_ma12")) /
      pl.when(pl.col("tn_sd12") > 0).then(pl.col("tn_sd12")).otherwise(1.0)).alias("z_vs_ma12"),
    (pl.col("share") - pl.col("share_ma6")).alias("share_desvio"),
    (pl.col("tn") / pl.when(pl.col("pico_hist") > 0).then(pl.col("pico_hist"))
                      .otherwise(1.0)).alias("ratio_pico_hist"),
])
P = P.with_columns(pl.col("m").map_elements(m_a_periodo, return_dtype=pl.Int64).alias("periodo"))

m_tr  = periodo_a_m(MESES_TRAIN_FIN)
m_val = [periodo_a_m(p) for p in MESES_VAL]
m_te  = [periodo_a_m(p) for p in MESES_TEST]
assert min(m_val) - m_tr >= H and min(m_te) - max(m_val) >= H, "falta gap"

D = P.drop_nulls(["y_tn","fase_fut"])
tr = D.filter(pl.col("m") <= m_tr)
va = D.filter(pl.col("m").is_in(m_val))
te = D.filter(pl.col("m").is_in(m_te))
print(f"train {tr.height:>7,}   val {va.height:>6,}   test {te.height:>6,}")

# Features BASE: sin ninguna referencia a la fase. Las arquitecturas la agregan.
BASE = [c for c in D.columns if c not in (
    "product_id","cat1","cat2","cat3","brand","m","periodo","m_nace",
    "y_tn","fase_fut","fase","fase_mercado","tn","share","tn_cat3")]
print(f"{len(BASE)} features base")

PARAMS = dict(objective="regression", metric="mae", verbosity=-1, n_jobs=-1,
              seed=SEMILLA, deterministic=True, force_row_wise=True,
              n_estimators=400, learning_rate=.05, num_leaves=63,
              min_child_samples=30, subsample=.8, subsample_freq=1, colsample_bytree=.8)

def wape(real, pred):
    real = np.asarray(real,float); pred = np.maximum(np.asarray(pred,float), 0)
    den = np.abs(real).sum()
    return float(np.abs(real-pred).sum()/den) if den else np.nan

def entrenar(feats, dtr=tr, target="y_tn", clasificador=False):
    Cls = lgb.LGBMClassifier if clasificador else lgb.LGBMRegressor
    pr = dict(PARAMS)
    if clasificador:
        pr.update(objective="multiclass", num_class=4, metric="multi_logloss")
    m = Cls(**pr)
    m.fit(dtr.select(feats).to_pandas(), dtr[target].to_numpy())
    return m

## 5 — Las cinco arquitecturas

In [ ]:
res = {}

# ── A: baseline sin fase ─────────────────────────────────────────────────
mA = entrenar(BASE)
res["A) baseline sin fase"] = (mA.predict(va.select(BASE).to_pandas()),
                               mA.predict(te.select(BASE).to_pandas()))

# ── B: fase observada en t (producto + mercado) como feature ─────────────
FB = BASE + ["fase", "fase_mercado"]
mB = entrenar(FB)
res["B) fase observada"] = (mB.predict(va.select(FB).to_pandas()),
                            mB.predict(te.select(FB).to_pandas()))

# ── C: fase PREDICHA para t+2, con out-of-fold ──────────────────────────
# CLAVE: las predicciones de la etapa 1 sobre TRAIN deben ser out-of-fold. Si se
# usan las in-sample, el clasificador acierta casi perfecto sobre las filas que ya
# vio, la etapa 2 aprende a confiar ciegamente en `fase_pred`, y en val/test -- donde
# el acierto real es mucho menor -- esa confianza se derrumba. Es el error clasico
# de stacking, y hace que la arquitectura se vea peor de lo que es.
FCLF = BASE + ["fase", "fase_mercado"]

meses_tr = sorted(tr["m"].unique().to_list())
K_FOLDS = 3
cortes = np.array_split(np.array(meses_tr), K_FOLDS + 1)   # el 1er bloque solo entrena

pr_tr = np.zeros((tr.height, 4))
m_tr_arr = tr["m"].to_numpy()
for k in range(1, K_FOLDS + 1):
    meses_fold = set(cortes[k].tolist())
    prev = tr.filter(pl.col("m") < min(meses_fold))        # solo el pasado del fold
    if prev.height < 500:
        continue
    clf_k = entrenar(FCLF, dtr=prev, target="fase_fut", clasificador=True)
    mask = np.isin(m_tr_arr, list(meses_fold))
    if mask.any():
        pr_tr[mask] = clf_k.predict_proba(
            tr.filter(pl.col("m").is_in(list(meses_fold))).select(FCLF).to_pandas())

# El primer bloque no tiene pasado suficiente: se le asigna la fase observada
# como one-hot, que es lo mejor disponible sin mirar el futuro.
sin_oof = pr_tr.sum(axis=1) == 0
if sin_oof.any():
    fobs = tr["fase"].to_numpy()[sin_oof]
    pr_tr[sin_oof, fobs] = 1.0
print(f"filas de train con prediccion out-of-fold: {(~sin_oof).sum():,} de {tr.height:,}")

# Para val y test: un clasificador entrenado con TODO el train (nunca vio val/test)
clf = entrenar(FCLF, target="fase_fut", clasificador=True)

def con_pred(df, pr=None):
    if pr is None:
        pr = clf.predict_proba(df.select(FCLF).to_pandas())
    return df.with_columns([pl.Series("fase_pred", pr.argmax(1).astype(np.int64))] +
                           [pl.Series(f"p_fase{k}", pr[:, k]) for k in range(4)])

tr_c, va_c, te_c = con_pred(tr, pr_tr), con_pred(va), con_pred(te)

acc_va      = float((va_c["fase_pred"] == va_c["fase_fut"]).mean())
acc_persist = float((va["fase"] == va["fase_fut"]).mean())
acc_tr_oof  = float((tr_c["fase_pred"] == tr_c["fase_fut"]).mean())
print(f"clasificador de fase   acierto val {acc_va:.3f}   vs persistir {acc_persist:.3f}")
print(f"                       acierto train out-of-fold {acc_tr_oof:.3f}")
print("Los dos aciertos tienen que parecerse. Si el de train fuera mucho mayor,")
print("seguiria habiendo fuga de la etapa 1 a la etapa 2.")

FC = FB + ["fase_pred"] + [f"p_fase{k}" for k in range(4)]
mC = entrenar(FC, dtr=tr_c)
res["C) fase predicha t+2"] = (mC.predict(va_c.select(FC).to_pandas()),
                               mC.predict(te_c.select(FC).to_pandas()))

In [ ]:
# ── D: un modelo por fase ────────────────────────────────────────────────
pv = np.zeros(va.height); pt = np.zeros(te.height)
fv = va["fase"].to_numpy(); ft = te["fase"].to_numpy()
for f in range(4):
    sub = tr.filter(pl.col("fase") == f)
    if sub.height < 300:                       # muy poco para un modelo propio
        m = mA
        print(f"   fase {FASES[f]:12s}: solo {sub.height} filas -> usa el modelo global")
    else:
        m = entrenar(BASE, dtr=sub)
        print(f"   fase {FASES[f]:12s}: modelo propio con {sub.height:,} filas")
    if (fv == f).any():
        pv[fv == f] = m.predict(va.filter(pl.col("fase") == f).select(BASE).to_pandas())
    if (ft == f).any():
        pt[ft == f] = m.predict(te.filter(pl.col("fase") == f).select(BASE).to_pandas())
res["D) un modelo por fase"] = (pv, pt)

R = pl.DataFrame([{"arquitectura": k,
                   "wape_val": round(wape(va["y_tn"], v[0]), 4),
                   "wape_test": round(wape(te["y_tn"], v[1]), 4)}
                  for k, v in res.items()])
R = R.with_columns((pl.col("wape_test") - pl.col("wape_val")).round(4).alias("brecha"))
base_w = R.filter(pl.col("arquitectura").str.starts_with("A"))["wape_test"][0]
R = R.with_columns((100*(base_w - pl.col("wape_test"))/base_w).round(2).alias("vs_baseline_%"))
print()
print(R.sort("wape_test"))

In [ ]:
d = R.sort("wape_test")
fig, ax = plt.subplots(figsize=(8.5, 3.2))
ys = list(range(d.height))
cols = [MUDO if n.startswith("A") else SERIE[0] for n in d["arquitectura"]]
ax.barh(ys, d["wape_test"], color=cols, height=.6)
for i, v in enumerate(d["wape_test"]):
    ax.annotate(f"{v:.4f}", (v, i), xytext=(6,0), textcoords="offset points",
                va="center", color=TINTA2, fontsize=9)
ax.set_yticks(ys); ax.set_yticklabels(d["arquitectura"])
ax.invert_yaxis(); ax.set_xlim(0, float(d["wape_test"].max())*1.2)
limpiar(ax, f"WAPE en test · horizonte {H}", x="WAPE")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
plt.tight_layout(); plt.show()

mejor = d.row(0, named=True)
print(f"VEREDICTO")
print(f"  mejor: {mejor['arquitectura']}  test {mejor['wape_test']:.4f} "
      f"({mejor['vs_baseline_%']:+.2f}% vs baseline)")
if abs(mejor["vs_baseline_%"]) < 1:
    print("  Diferencia menor al 1%: con un test de un mes eso es ruido.")
    print("  Detectar la fase por separado NO aporta sobre darle las features crudas")
    print("  al GBM, que ya puede inferir el regimen solo.")
else:
    print("  La etapa de fase aporta. Ojo igual con el tamanio del test.")

## 6 — Dónde gana cada arquitectura

El promedio esconde lo importante. Igual que en `05`, lo que decide si esto sirve es
si la ventaja se **concentra** en algún estado: una mejora del 15% en lanzamiento
vale aunque el promedio no se mueva, porque se puede rutear.

In [ ]:
ev = te.select("fase","fase_mercado","y_tn").to_pandas()
for k, v in res.items():
    ev[k] = v[1]

filas = []
for f, g in ev.groupby("fase"):
    if len(g) < 20:
        continue
    fila = {"fase": FASES[f], "n": len(g)}
    for k in res:
        fila[k.split(")")[0]] = round(wape(g["y_tn"], g[k]), 4)
    filas.append(fila)
print("=== POR FASE DEL PRODUCTO ===")
print(pl.DataFrame(filas))

filas = []
for (f, fm), g in ev.groupby(["fase","fase_mercado"]):
    if len(g) < 30:
        continue
    fila = {"estado": f"{FASES[f]} / {MFASES[fm]}", "n": len(g)}
    for k in res:
        fila[k.split(")")[0]] = round(wape(g["y_tn"], g[k]), 4)
    filas.append(fila)
print()
print("=== POR ESTADO CONJUNTO ===")
print(pl.DataFrame(filas))
print()
print("ADVERTENCIA: test = 1 mes. Con n de dos digitos por celda, estas diferencias")
print("son indicativas. Repetir con walk-forward antes de decidir.")

## 7 — Exportar las etiquetas de estado

In [ ]:
export = (P.select("product_id","periodo","fase","fase_mercado","pend3","cat3_pend3",
                   "edad","ratio_pico_hist","z_vs_ma12","share_desvio")
            .with_columns(
                (pl.col("fase")*3 + pl.col("fase_mercado")).alias("estado_conjunto")))

out = DIR_OUT / "features_fase_estado.parquet"
export.write_parquet(out)
print(f"Guardado: {out}")
print(f"{export.height:,} filas x {export.width} columnas")
print(export.columns)
print()
print("Todas causales: fase y fase_mercado usan el maximo y la pendiente HASTA ese mes.")
print("estado_conjunto = fase*3 + fase_mercado, para usarlo como una sola categorica.")

### Cómo llevarlo al pipe

```python
fases = pl.read_parquet(RUTA_FE / "features_fase_estado.parquet")
df_norm = df_norm.join(fases, on=["product_id", "periodo"], how="left")
```

Y en `03_Optuna`, declarar las categóricas y marcar el experimento:

```python
'cols_categoricas': ['cat1','cat2','cat3','brand','fase','fase_mercado','estado_conjunto'],
'sufijo': 'conFase',
```

**Una expectativa realista antes de correrlo**: la fase se calcula a partir de la
pendiente de la media móvil, y el GBM ya recibe los lags con los que podría
derivarla solo. Es probable que la ganancia como *feature* sea chica. Donde la fase
tiene valor demostrado es como **router** — decidir qué arquitectura usar, que fue el
+15,7% en lanzamiento de `05`. Eso no es algo que el GBM pueda hacer por su cuenta,
porque es una decisión sobre el modelo, no sobre los datos.